# JetBot - Data collection

In this notebook we'll collect training data for CNN VAE. The training data save to dataset directory. Need USB gamepad for running.

## Import module

Import required module for this notebook. MobileController module is own module. This module implements Jetbot steering control.


In [2]:
!apt-get update && apt-get install -y --no-install-recommends python3-gi gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly gstreamer1.0-libav

Get:1 https://deb.nodesource.com/node_10.x bionic InRelease [4584 B]
Get:2 https://deb.nodesource.com/node_10.x bionic/main arm64 Packages [767 B]  
Get:3 http://ports.ubuntu.com/ubuntu-ports bionic InRelease [242 kB]
Get:4 http://ports.ubuntu.com/ubuntu-ports bionic-updates InRelease [102 kB]
Get:5 http://ports.ubuntu.com/ubuntu-ports bionic-backports InRelease [102 kB]
Get:6 http://ports.ubuntu.com/ubuntu-ports bionic-security InRelease [102 kB]
Get:7 http://ports.ubuntu.com/ubuntu-ports bionic/restricted arm64 Packages [572 B]
Get:8 http://ports.ubuntu.com/ubuntu-ports bionic/multiverse arm64 Packages [153 kB]
Get:9 http://ports.ubuntu.com/ubuntu-ports bionic/main arm64 Packages [1285 kB]
Get:10 http://ports.ubuntu.com/ubuntu-ports bionic/universe arm64 Packages [11.0 MB]
Get:11 http://ports.ubuntu.com/ubuntu-ports bionic-updates/main arm64 Packages [2310 kB]
Get:12 http://ports.ubuntu.com/ubuntu-ports bionic-updates/restricted arm64 Packages [6338 B]
Get:13 http://ports.ubuntu.com/

In [5]:
!apt-get update && apt-get install -y python3-gst-1.0

Hit:1 https://deb.nodesource.com/node_10.x bionic InRelease
Hit:2 http://ports.ubuntu.com/ubuntu-ports bionic InRelease                    
Hit:3 http://ports.ubuntu.com/ubuntu-ports bionic-updates InRelease
Hit:4 http://ports.ubuntu.com/ubuntu-ports bionic-backports InRelease
Hit:5 http://ports.ubuntu.com/ubuntu-ports bionic-security InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  gir1.2-gst-plugins-base-1.0 gir1.2-gstreamer-1.0
The following NEW packages will be installed:
  gir1.2-gst-plugins-base-1.0 gir1.2-gstreamer-1.0 python3-gst-1.0
0 upgraded, 3 newly installed, 0 to remove and 140 not upgraded.
Need to get 168 kB of archives.
After this operation, 1950 kB of additional disk space will be used.
Get:1 http://ports.ubuntu.com/ubuntu-ports bionic-updates/main arm64 gir1.2-gstreamer-1.0 arm64 1.14.5-0ubuntu1~18.04.2 [71.6 kB]
Get:2 http://p

In [6]:
import os
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
from mobile import MobileController

In [ ]:
!apt-get update && apt-get install -y curl
!apt-get update && apt-get install -y nodejs
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

Hit:1 https://deb.nodesource.com/node_10.x bionic InRelease
Hit:2 http://ports.ubuntu.com/ubuntu-ports bionic InRelease                    
Hit:3 http://ports.ubuntu.com/ubuntu-ports bionic-updates InRelease
Hit:4 http://ports.ubuntu.com/ubuntu-ports bionic-backports InRelease
Hit:5 http://ports.ubuntu.com/ubuntu-ports bionic-security InRelease
Reading package lists... Done                     
Reading package lists... Done
Building dependency tree       
Reading state information... Done
curl is already the newest version (7.58.0-2ubuntu3.24).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.
Hit:1 https://deb.nodesource.com/node_10.x bionic InRelease
Hit:2 http://ports.ubuntu.com/ubuntu-ports bionic InRelease                 
Hit:3 http://ports.ubuntu.com/ubuntu-ports bionic-updates InRelease
Hit:4 http://ports.ubuntu.com/ubuntu-ports bionic-backports InRelease
Hit:5 http://ports.ubuntu.com/ubuntu-ports bionic-security InRelease
Reading package lists... Done
Reading pa

## Activation Gamepad.
This step is similar to "Teleoperation" task. In this task, we will use gamepad controller to collect training data.

The first thing we want to do is create an instance of the Controller widget, which we'll use to control jetbot with speed and steering. The Controller widget takes a index parameter, which specifies the number of the controller. This is useful in case you have multiple controllers attached, or some gamepads appear as multiple controllers. To determine the index of the controller you're using,

Visit http://html5gamepad.com. Press buttons on the gamepad you're using Remember the index of the gamepad that is responding to the button presses Next, we'll create and display our controller using that index.


In [15]:
controller = widgets.Controller(index=0)
display(controller)

Controller()

Now we can look Button assignment of gamepad. let check your need key assignment.

* SPEED is left side joystick uptodown almost is joys 1
* STERING is right side joystick left-right almost is joys2
* RECORDING is right side  trigger almost is buttons 5


In [8]:
SPEED=1
STERING=2
RECORDING=5

## Steering Control

Steering control need to "wheel track" parameter. Wheel track is between distance left-right wheels.
We'll measure wheel track  measure in cm.

In [4]:
robot = Robot()
WHEEL_TRACK = 10 #cm
mobile_controller = MobileController(WHEEL_TRACK,robot)

We'll run next block. and We can controll jetbot with gamepad.

In [5]:
speed = widgets.FloatSlider(min=-1.0, max=1.0, description='speed')
steering = widgets.FloatSlider(min=-1.0, max=1.0, description='steering')


speed_link = traitlets.dlink((controller.axes[SPEED], 'value'), (mobile_controller, 'speed'), transform=lambda x: -x)
steering_link = traitlets.dlink((controller.axes[STERING], 'value'), (mobile_controller, 'radius'))
traitlets.dlink((mobile_controller, 'speed'), (speed,'value'))
traitlets.dlink((mobile_controller, 'radius'), (steering, 'value'))


## Initialize Camera

Next is initializing camera module. Image size is 320 x 240. Frame rate is about 27Hz. We'll save image in camera observer method. camera observer method can get image per frame rate. Thus, frame rate is decide to image save interval.

In [6]:
camera = Camera.instance(width=320, height=240)
image = widgets.Image(format='jpeg', width=320, height=240)
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)


## UI Widget

We can check Gamepad value and Image. 

In [7]:
DATASET_DIR = 'dataset'
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Directories not created becasue they already exist')

dataset=DATASET_DIR
layout = widgets.Layout(width='100px', height='64px')
count_box   = widgets.IntText(layout=layout, value=len(os.listdir(dataset)))
count_label = widgets.Label(layout=layout, value='Number image:')
count_panel = widgets.HBox([count_label,count_box])

panel = widgets.VBox([count_panel, speed, steering])
display(widgets.HBox([panel,image]))


Directories not created becasue they already exist


## Set callback for collect the training data.

```save_record``` is callback for training data. The method set to camera observer. This callback saving the image that contain speed and steering in file name to DATASET_DIR. When holding ```R``` button, this method recording training data. You can check number of training data with ```Number image text box```.

In [8]:

import os
from uuid import uuid1

def save_record(change):                
    global controller, speed, steering, mobile_controller
    if controller.buttons[RECORDING].value==1.0:
        
        image_name = '{:.02f}_{:.02f}_{}.jpg'.format(mobile_controller.speed, mobile_controller.radius,uuid1())
        image_path = os.path.join(DATASET_DIR, image_name)
        save_image=bgr8_to_jpeg(change['new'])
        with open(image_path, 'wb') as f:
            f.write(save_image)
        count_box.value = len(os.listdir(dataset)) 


save_record({'new': camera.value})
camera.observe(save_record, names='value')

## Cleanup
After collecting enough data. cleanup camera observer and stop all motor.

In [9]:
camera.unobserve(save_record, names='value')
camera_link.unlink()
speed_link.unlink()
steering_link.unlink()
robot.stop()

## Cleate dataset.zip file

In [ ]:
import datetime
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q jetbot_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}